# N031 · 前缀聚合与区间查询

**目标：** 明确半开区间并区分可逆和不可逆聚合。

**先修：** N017, N013。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 前缀和/异或；加法逆元；前后缀乘积；prefix min不能直接相减。

## 从问题到算法

区间求和反复查询时，先保存每个位置前的累计值。采用半开区间 [l,r)，让空区间和、整段和和长度 r−l 保持统一。除自身乘积不能在含零时直接除总乘积；用左侧乘积与右侧乘积相乘，不需要除法。

$$
P[0]=0,\quad P[i+1]=P[i]+a_i,\quad S(l,r)=P[r]-P[l].
$$

## 最小实现

**本章接口：** `PrefixSum`、`product_except_self(nums)`

In [1]:
class PrefixSum:
    def __init__(self,nums):
        self.prefix=[0]
        for x in nums: self.prefix.append(self.prefix[-1]+x)
    def query(self,left,right):
        if not 0<=left<=right<len(self.prefix): raise IndexError('invalid half-open interval')
        return self.prefix[right]-self.prefix[left]

def product_except_self(nums):
    answer=[1]*len(nums); product=1
    for i,x in enumerate(nums):
        answer[i]=product; product*=x
    product=1
    for i in range(len(nums)-1,-1,-1):
        answer[i]*=product; product*=nums[i]
    return answer

## 正确性、前提与复杂度

两前缀相减取消共同的[0,l)部分。乘积首遍记录严格左侧，逆遍乘严格右侧，两部分互不重叠且恰好覆盖除当前位置之外的全部索引。

**代价：** 前缀构建 O(n)、查询 O(1)、空间 O(n)；乘积 O(n) 时间、O(1) 辅助空间（不计输出）。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,2,3,4]; p=PrefixSum(a)
show_table(['位置i','P[i]','前缀区间'],[(i,v,f'[0,{i})') for i,v in enumerate(p.prefix)])
show_table(['输入','除自身乘积'],[(a,product_except_self(a)),([0,2,3],product_except_self([0,2,3]))])

位置i,P[i],前缀区间
0,0,"[0,0)"
1,1,"[0,1)"
2,3,"[0,2)"
3,6,"[0,3)"
4,10,"[0,4)"


输入,除自身乘积
"[1, 2, 3, 4]","[24, 12, 8, 6]"
"[0, 2, 3]","[6, 0, 0]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
p=PrefixSum([1,2,3]); assert p.query(1,3)==5 and p.query(2,2)==0
assert product_except_self([1,2,3,4])==[24,12,8,6]
assert product_except_self([0,2,3])==[6,0,0]
assert product_except_self([0,2,0])==[0,0,0]
assert product_except_self([])==[]
for l in range(4):
    for r in range(l,4): assert p.query(l,r)==sum([1,2,3][l:r])
print('N031: 所有本章断言通过')

N031: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 写出前缀异或查询。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 构造前缀最小值无法恢复任意区间最小值的反例。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 303. Range Sum Query - Immutable（canonical）
- 238. Product of Array Except Self（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。